<a href="https://colab.research.google.com/github/ritan612/S-AES-OFB/blob/main/IN410.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Initialization

## Initialization Vector(IV) And Input the Key

In [ ]:
import numpy as np
import random

def initializing_IV():
  IV = random.getrandbits(16)
  # Le '016b' signifie : format binaire (b), sur 16 caractères (16), rempli de zéros (0)
  print(f"Initialization Vector:{IV:016b}")
  return IV

def initializing_S_AES_key():
  while True:
      key = input("Enter 16-bit key: ")
      # Check length and allowed characters
      if len(key) == 16 and all(bit in '01' for bit in key):
          break
      else:
          print("Invalid input! Please enter exactly 16 bits (only 0 and 1).")

  # Convert to array (list of integers)
  key_array = [int(bit) for bit in key]

  print("Valid key:", key)
  print("Key as array:", key_array)

Enter 16-bit key: 1111111100000000
Valid key: 1111111100000000
Key as array: [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]


## Upload the file

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from google.colab import files
uploaded = files.upload()
uploaded_path = list(uploaded.keys())[0]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Saving Heelo World.txt to Heelo World (2).txt


## Convert the file into the Plaintext (bits array)

In [ ]:
def file_to_bits_array(file_path, chunk_size=8192):
  bits_array = []
  with open(file_path, "rb") as f:
      while chunk := f.read(chunk_size):
          for byte in chunk:
              bits = f"{byte:08b}"
              for bit in bits:
                  bits_array.append(int(bit))
  return bits_array

bits = file_to_bits_array(uploaded_path)

print(bits[:64])  # first 64 bits
print(len(bits))   # total number of bits


[0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1]
88


## Split The Plaintext into 16-bit Blocks

In [ ]:
from itertools import batched

def batched_with_padding(iterable, batch_size, fill_value=0):
    for batch in batched(iterable, batch_size):
        yield batch + (fill_value,) * (batch_size - len(batch))

batches = {}

for i, batch in enumerate(batched_with_padding(bits, 16), start=1):
    name = f"p{i}"
    batches[name] = batch
    print(name, "=", batch)

def bits_to_int(bits):
    return int("".join(str(b) for b in bits), 2)

p1 = (0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1)
p2 = (0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0)
p3 = (0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0)
p4 = (0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1)
p5 = (0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0)
p6 = (0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0)


# Simplified AES (S-AES) implementation

## SBOX and Inverse SBOX

In [ ]:
SBOX = {0x0:0x9,0x1:0x4,0x2:0xA,0x3:0xB,
        0x4:0xD,0x5:0x1,0x6:0x8,0x7:0x5,
        0x8:0x6,0x9:0x2,0xA:0x0,0xB:0x3,
        0xC:0xC,0xD:0xE,0xE:0xF,0xF:0x7}

INV_SBOX = {v:k for k,v in SBOX.items()}


## Key Generation

In [ ]:
def sub_nib(b): return (SBOX[b>>4]<<4) | SBOX[b&0xF] #t2asem el bytes 2osmen w bet tabe2 3layoun el Sbox b>>4 awal nibble b&0xF ekhir nibble
def rot_nib(b): return ((b&0xF)<<4) | (b>>4) #tbadel el nosen

def key_expansion(key): #generate sub-key
  W0 = (key>>8)&0xFF # awal 8 w byedman enno 8 bits
  W1 = key&0xFF #ekhir 8 bits
  W2 = W0 ^ 0x80 ^ sub_nib(rot_nib(W1)) #RCon[1]=80 (10000000), ^ =XOR
  W3 = W2 ^ W1
  W4 = W2 ^ 0x30 ^ sub_nib(rot_nib(W3)) #RCon[2]=30 (00110000)
  W5 = W4 ^ W3
  K0 = (W0<<8)|W1
  K1 = (W2<<8)|W3
  K2 = (W4<<8)|W5
  print(f"Key0 = {K0:016b}")
  print(f"Key1 = {K1:016b}")
  print(f"Key2 = {K2:016b}")
  return K0,K1,K2


## Encryption

In [ ]:

def add_round_key(state,key): return state ^ key #Plaintext XOR Key0

def sub_nibbles(state): # n2asem kel nibble la 7al (4 nibbles)
    return (SBOX[(state>>12)&0xF]<<12 |
            SBOX[(state>>8)&0xF]<<8 |
            SBOX[(state>>4)&0xF]<<4 |
            SBOX[state&0xF])

def shift_rows(state):
    # swap 2nd and 4th nibble
    n0=(state>>12)&0xF; n1=(state>>8)&0xF
    n2=(state>>4)&0xF;  n3=state&0xF
    return (n0<<12)|(n3<<8)|(n2<<4)|(n1)

def mix_columns(state):
    # simplified GF(2^4) multiplication
    def mult(a,b):
        res=0
        while b: #la tsir b = 0000
            if b&1: res^=a # res = b XOR a
            a<<=1 # shift a ex.: 0100 -> 1000 / 1000 -> 10000 overflow
            if a&0x10: a^=0x13 #x⁴ + x + 1  → 0x13 (10011) ex.: 10000 XOR 10011 = 00011
            b>>=1 # shift b ex.: 0011 -> 0001
        return res&0xF
    n0=(state>>12)&0xF; n1=(state>>8)&0xF # n2asem el nibbles
    n2=(state>>4)&0xF;  n3=state&0xF
    r0=n0 ^ mult(4,n1) #r0 = n0 XOR (4 x n1)
    r1=mult(4,n0) ^ n1 # r1= (4 x n0) XOR n1
    r2=n2 ^ mult(4,n3)
    r3=mult(4,n2) ^ n3
    return (r0<<12)|(r1<<8)|(r2<<4)|r3

def encrypt(plaintext,key):
    K0,K1,K2=key_expansion(key)
    print(f"Plaintext = {plaintext:016b}")
    state=add_round_key(plaintext,K0)
    print(f"After AddRoundKey0 = {state:016b}")
    state=sub_nibbles(state)
    print(f"After SubNibbles1 = {state:016b}")
    state=shift_rows(state)
    print(f"After ShiftRows1   = {state:016b}")
    state=mix_columns(state)
    print(f"After MixColumns1  = {state:016b}")
    state=add_round_key(state,K1)
    print(f"After AddRoundKey1 = {state:016b}")
    state=sub_nibbles(state)
    print(f"After SubNibbles2  = {state:016b}")
    state=shift_rows(state)
    print(f"After ShiftRows2   = {state:016b}")
    state=add_round_key(state,K2)
    print(f"Ciphertext         = {state:016b}")
    return state


## Decryption

In [ ]:
def inv_sub_nibbles(state):
    return (INV_SBOX[(state>>12)&0xF]<<12 |
            INV_SBOX[(state>>8)&0xF]<<8 |
            INV_SBOX[(state>>4)&0xF]<<4 |
            INV_SBOX[state&0xF])

def inv_shift_rows(state):
    # same as shift_rows for 2nd and 4th nibble swap
    n0=(state>>12)&0xF; n1=(state>>8)&0xF
    n2=(state>>4)&0xF;  n3=state&0xF
    return (n0<<12)|(n3<<8)|(n2<<4)|(n1)

def inv_mix_columns(state):
    def mult(a,b):
        res=0
        while b:
            if b&1: res^=a
            a<<=1
            if a&0x10: a^=0x13
            b>>=1
        return res&0xF
    n0=(state>>12)&0xF; n1=(state>>8)&0xF
    n2=(state>>4)&0xF;  n3=state&0xF
    r0=mult(9,n0)^mult(2,n1)
    r1=mult(2,n0)^mult(9,n1)
    r2=mult(9,n2)^mult(2,n3)
    r3=mult(2,n2)^mult(9,n3)
    return (r0<<12)|(r1<<8)|(r2<<4)|r3

def decrypt(ciphertext,key):
    K0,K1,K2=key_expansion(key)
    print(f"Ciphertext = {ciphertext:016b}")
    state=add_round_key(ciphertext,K2)
    print(f"After AddRoundKey2 = {state:016b}")
    state=inv_shift_rows(state)
    print(f"After InvShiftRows2 = {state:016b}")
    state=inv_sub_nibbles(state)
    print(f"After InvSubNibbles2= {state:016b}")
    state=add_round_key(state,K1)
    print(f"After AddRoundKey1  = {state:016b}")
    state=inv_mix_columns(state)
    print(f"After InvMixColumns1= {state:016b}")
    state=inv_shift_rows(state)
    print(f"After InvShiftRows1 = {state:016b}")
    state=inv_sub_nibbles(state)
    print(f"After InvSubNibbles1= {state:016b}")
    state=add_round_key(state,K0)
    print(f"Recovered Plaintext = {state:016b}")
    return state



# Implement OFB With S-AES

In [ ]:
# Step 1: IV
iv = initializing_IV()

# make sure that key is an int
if isinstance(key, str):
    key = int(key, 2)

cipher_blocks = {}
O = iv

for i, (name, bits_block) in enumerate(batches.items(), start=1):

    # Step 2: Encrypt Using S-AES
    O = encrypt(O, key)
    print(f"O{i} = {O:016b}")

    # Step 3: Convert Plaintext from bits to int
    P = bits_to_int(bits_block)
    print(f"{name} = {P:016b}")

    # Step 4: XOR with plaintext
    C = P ^ O
    cipher_blocks[f"C{i}"] = C

    print(f"C{i} = {C:016b}\n")

print("\nFinal Ciphertext:")
for name, value in cipher_blocks.items():
    print(f"{name} = {value:016b}")

print("\n------Decryption--------")
decrypted_blocks = {}
O = iv

for i, (name, C) in enumerate(cipher_blocks.items(), start=1):
    O = encrypt(O, key)
    P = C ^ O # XOR is reversible
    decrypted_blocks[f"P{i}"] = P
    print(f"P{i} = {P:016b}\n")

Initialization Vector:1001100001101000
Key0 = 1111111100000000
Key1 = 1110011011100110
Key2 = 0101100110111111
Plaintext = 1001100001101000
After AddRoundKey0 = 0110011101101000
After SubNibbles1 = 1000010110000110
After ShiftRows1   = 1000011010000101
After MixColumns1  = 0011000011110011
After AddRoundKey1 = 1101011000010101
After SubNibbles2  = 1110100001000001
After ShiftRows2   = 1110000101001000
Ciphertext         = 1011100011110111
O1 = 1011100011110111
p1 = 0100100001100101
C1 = 1111000010010010

Key0 = 1111111100000000
Key1 = 1110011011100110
Key2 = 0101100110111111
Plaintext = 1011100011110111
After AddRoundKey0 = 0100011111110111
After SubNibbles1 = 1101010101110101
After ShiftRows1   = 1101010101110101
After MixColumns1  = 1010010000001010
After AddRoundKey1 = 0100001011101100
After SubNibbles2  = 1101101011111100
After ShiftRows2   = 1101110011111010
Ciphertext         = 1000010101000101
O2 = 1000010101000101
p2 = 0110110001101100
C2 = 1110100100101001

Key0 = 111111110000